# 02 — Automate the event research

Applies the same rules used in the manual review to every candidate, using free public data:
- **SEC EDGAR** for company identity (CIK), filings, and shares outstanding
- **yfinance** for prices and the trading calendar

Rows already researched by hand override the automated values. Anything the rules can't settle is marked `review` for a manual check.

**Before running:** in Google Sheets, File → Download → CSV, and save it as `data/processed/review_sheet_manual.csv`.

In [1]:
import json, re, time, difflib
from pathlib import Path
import numpy as np
import pandas as pd
import requests
import yfinance as yf

# SEC requires a User-Agent with a name and email
SEC_HEADERS = {"User-Agent": "Chengzhe Jiang chengzhe@berkeley.edu"}
DATA = Path("../data")
CACHE = DATA / "cache"
CACHE.mkdir(parents=True, exist_ok=True)

def sec_get_json(url, cache_name):
    """Download a JSON file from SEC once, then reuse the saved copy."""
    path = CACHE / cache_name
    if path.exists():
        return json.loads(path.read_text())
    r = requests.get(url, headers=SEC_HEADERS, timeout=60)
    r.raise_for_status()
    path.write_text(r.text)
    time.sleep(0.15)  # SEC limit: 10 requests per second
    return r.json()

## Step 1: Load the candidates and the hand-researched rows

In [4]:
review = pd.read_csv(DATA / "processed/review_sheet.csv", parse_dates=["FDA Approval Date"])

manual_path = DATA / "processed/review_sheet_manual.csv"
if manual_path.exists():
    manual = pd.read_csv(manual_path)
    manual["FDA Approval Date"] = pd.to_datetime(manual["FDA Approval Date"], errors="coerce")
    keep = ["Proprietary Name", "FDA Approval Date", "ticker", "market_cap_usd_m",
            "event_day0", "include", "exclude_reason"]
    review = review[[c for c in review.columns if c not in keep[2:]]].merge(
        manual[keep], on=["Proprietary Name", "FDA Approval Date"], how="left")
    review["event_day0"] = pd.to_datetime(review["event_day0"], errors="coerce")
else:
    print("No manual sheet found; running fully automated.")
    for c in ["ticker", "market_cap_usd_m", "event_day0", "include", "exclude_reason"]:
        review[c] = np.nan

done = review["include"].isin(["Y", "N"])
print(len(review), "candidates;", done.sum(), "already researched by hand")

226 candidates; 13 already researched by hand


## Step 2: Match each applicant to an SEC-registered company

SEC's ticker list only contains companies that are **currently** listed. Applicants that don't match are usually private, foreign, a subsidiary, or acquired since approval; they're marked for review.

In [7]:
pd.set_option("display.max_rows", 200)

tick = sec_get_json("https://www.sec.gov/files/company_tickers.json", "company_tickers.json")
sec_cos = pd.DataFrame(tick.values())

SUFFIXES = r"\b(inc|incorporated|corp|corporation|co|company|ltd|limited|llc|lp|plc|sa|ag|nv|bv|ab|as|gmbh|spa|usa|us|holdings?|group)\b"
def norm(name):
    s = re.sub(r"[^a-z0-9 ]", " ", str(name).lower())
    s = re.sub(SUFFIXES, " ", s)
    return re.sub(r"\s+", " ", s).strip()

sec_cos["norm"] = sec_cos["title"].apply(norm)
by_name = sec_cos.groupby("norm").first()

def match_company(applicant):
    n = norm(applicant)
    if not n:
        return np.nan, np.nan, np.nan, 0.0
    if n in by_name.index:  # exact match after cleaning the name
        r = by_name.loc[n]
        return r["cik_str"], r["ticker"], r["title"], 1.0
    for c in difflib.get_close_matches(n, by_name.index, n=5, cutoff=0.85):
        if c.split()[0] == n.split()[0]:  # the distinctive first word must match
            r = by_name.loc[c]
            return r["cik_str"], r["ticker"], r["title"], round(difflib.SequenceMatcher(None, n, c).ratio(), 2)
    return np.nan, np.nan, np.nan, 0.0

# Remove columns from any earlier run, so rerunning this cell is safe
review = review.drop(columns=["cik", "auto_ticker", "sec_title", "match_score"], errors="ignore")

m = pd.DataFrame(review["Applicant"].apply(match_company).tolist(),
                 columns=["cik", "auto_ticker", "sec_title", "match_score"], index=review.index)
review = pd.concat([review, m], axis=1)
print("Matched:", review["cik"].notna().sum(), "of", len(review))

# Check these by eye: each should be the same company, written slightly differently
review.loc[(review["match_score"] > 0) & (review["match_score"] < 1),
           ["Applicant", "sec_title", "auto_ticker", "match_score"]]

Matched: 79 of 226


,Applicant,sec_title,auto_ticker,match_score
105,ADC Therapeutics S.A.,ADC Therapeutics SA,ADCT,0.89
108,Zealand Pharma A/S,Zealand Pharma A/S/ADR,ZLDPF,0.90
130,Dr. Reddy’s Laboratories,DR REDDYS LABORATORIES LTD,RDY,0.98


## Step 3: Trading calendar and day 0

**Automated rule:** day 0 = the first trading day on or after the FDA approval date. Rows with a hand-researched day 0 keep it.

This can be off by one day for after-close announcements, which is why the main window is **[-1, +1]**: a three-day window captures the reaction either way.

In [11]:
spx = yf.download("^GSPC", start="2013-01-01", progress=False, auto_adjust=True)
tdays = pd.DatetimeIndex(spx.index).tz_localize(None).normalize()

def tday_index(d):
    return int(tdays.searchsorted(pd.Timestamp(d).normalize()))  # first trading day on/after d

review["day0"] = review["event_day0"].fillna(review["FDA Approval Date"])
review["day0_idx"] = review["day0"].apply(tday_index)
review["day0"] = tdays[review["day0_idx"].clip(upper=len(tdays) - 1)]
review["day0_source"] = np.where(review["event_day0"].notna(), "manual", "fda_date")

## Step 4: Flag possible confounding events from SEC filings

For each event, list filings within **±5 trading days** that signal other material news:
- 8-K **Item 2.02** (earnings), **1.01** (material agreement), **2.01** (acquisition), **3.02** (equity sale), **5.02** (executive change)
- **424B** prospectus filings (possible stock offering)

**Limitation:** news released only as a press release, with no filing, isn't caught here.

In [ ]:
def load_filings(cik):
    cik10 = str(int(cik)).zfill(10)
    sub = sec_get_json(f"https://data.sec.gov/submissions/CIK{cik10}.json", f"sub_{cik10}.json")
    frames = [pd.DataFrame(sub["filings"]["recent"])]
    for f in sub["filings"].get("files", []):  # older filings live in extra pages
        frames.append(pd.DataFrame(sec_get_json(f"https://data.sec.gov/submissions/{f['name']}", f["name"])))
    df = pd.concat(frames, ignore_index=True).reindex(columns=["form", "filingDate", "items"])
    df["filingDate"] = pd.to_datetime(df["filingDate"])
    return df

filings = {}
for cik in review["cik"].dropna().unique():
    try:
        filings[cik] = load_filings(cik)
    except Exception as e:
        print("Could not load filings for CIK", cik, "-", e)
print("Filings loaded for", len(filings), "companies")

FLAG_ITEMS = {"2.02": "earnings", "1.01": "material agreement", "2.01": "acquisition",
              "3.02": "equity sale", "5.02": "exec change"}

def edgar_flags(cik, d0_idx):
    if cik not in filings:
        return np.nan
    lo = tdays[max(d0_idx - 5, 0)]
    hi = tdays[min(d0_idx + 5, len(tdays) - 1)]
    f = filings[cik]
    w = f[(f["filingDate"] >= lo) & (f["filingDate"] <= hi)]
    flags = []
    for _, r in w.iterrows():
        form = str(r["form"])
        if form.startswith("8-K"):
            hits = [FLAG_ITEMS[i.strip()] for i in str(r["items"]).split(",") if i.strip() in FLAG_ITEMS]
            if hits:
                flags.append(f"{r['filingDate'].date()} 8-K ({', '.join(hits)})")
        elif form.startswith("424B"):
            flags.append(f"{r['filingDate'].date()} {form} (possible offering)")
    return "; ".join(flags)

review["edgar_flags"] = [edgar_flags(c, i) for c, i in zip(review["cik"], review["day0_idx"])]
review["edgar_flags"].replace("", np.nan).notna().sum()

## Step 5: Market cap on day -1 and price-data check

**Market cap = day -1 closing price × shares outstanding**, using the latest share count the company reported to SEC *before* the event (no look-ahead).

Yahoo's historical prices are adjusted for later stock splits, which is common with small biotechs (reverse splits). The code undoes that adjustment so the price matches the share count from that time.

In [ ]:
def shares_before(cik, date):
    cik10 = str(int(cik)).zfill(10)
    try:
        facts = sec_get_json(f"https://data.sec.gov/api/xbrl/companyfacts/CIK{cik10}.json", f"facts_{cik10}.json")
        rows = pd.DataFrame(facts["facts"]["dei"]["EntityCommonStockSharesOutstanding"]["units"]["shares"])
    except Exception:
        return np.nan
    rows["filed"] = pd.to_datetime(rows["filed"])
    rows["end"] = pd.to_datetime(rows["end"])
    rows = rows[rows["filed"] < date]
    if rows.empty:
        return np.nan
    latest = rows.sort_values(["end", "filed"]).iloc[-1]["accn"]
    return rows.loc[rows["accn"] == latest, "val"].sum()  # sums share classes in that filing

price_cache = {}
def prices(ticker):
    if ticker not in price_cache:
        t = yf.Ticker(ticker)
        h = t.history(start="2013-01-01", auto_adjust=False)
        if not h.empty:
            h.index = h.index.tz_localize(None).normalize()
        s = t.splits
        if len(s):
            s.index = s.index.tz_localize(None).normalize()
        price_cache[ticker] = (h, s)
    return price_cache[ticker]

def mcap_and_coverage(row):
    tkr = row["ticker"] if isinstance(row["ticker"], str) and row["ticker"] not in ("N/A", "none") else row["auto_ticker"]
    if not isinstance(tkr, str) or pd.isna(row["cik"]):
        return pd.Series([np.nan, 0, 0])
    try:
        h, splits = prices(tkr)
    except Exception:
        return pd.Series([np.nan, 0, 0])
    if h.empty:
        return pd.Series([np.nan, 0, 0])
    d0 = row["day0"]
    before, after = (h.index < d0).sum(), (h.index >= d0).sum()
    if before == 0:
        return pd.Series([np.nan, before, after])
    dm1 = h.index[h.index < d0][-1]
    later = splits[splits.index > dm1].prod() if len(splits) else 1.0
    raw_close = h.loc[dm1, "Close"] * later
    sh = shares_before(row["cik"], d0)
    mcap = raw_close * sh / 1e6 if pd.notna(sh) else np.nan
    return pd.Series([mcap, before, after])

review[["auto_mcap_m", "rows_before", "rows_after"]] = review.apply(mcap_and_coverage, axis=1)
review["auto_mcap_m"].describe()

## Step 6: Apply the rules

Same rules as the manual review, applied in order. Borderline market caps (within 15% of a cutoff) go to manual review.

In [ ]:
def auto_status(r):
    if pd.isna(r["cik"]):
        return "review: no SEC match (private, foreign, subsidiary, or delisted)"
    if r["rows_before"] < 250 or r["rows_after"] < 6:
        return "exclude: insufficient price data"
    mc = r["auto_mcap_m"]
    if pd.isna(mc):
        return "review: market cap not computed"
    if 255 <= mc <= 345 or 8500 <= mc <= 11500:
        return "review: market cap near cutoff"
    if mc < 300:
        return "exclude: market cap < $300M"
    if mc > 10000:
        return "exclude: market cap > $10B"
    if isinstance(r["edgar_flags"], str) and r["edgar_flags"]:
        return "review: possible confounding event"
    return "include"

review["auto_status"] = review.apply(auto_status, axis=1)
print(review["auto_status"].value_counts())

## Step 7: Validate against the hand-researched rows

Before trusting the automation, check how often it agrees with the decisions made by hand.

In [ ]:
check = review[review["include"].isin(["Y", "N"])].copy()
check["auto_says"] = np.select(
    [check["auto_status"] == "include", check["auto_status"].str.startswith("exclude")],
    ["Y", "N"], default="review")
print(pd.crosstab(check["include"], check["auto_says"], rownames=["by hand"], colnames=["automated"]))
check[check["include"] != check["auto_says"]][
    ["Proprietary Name", "Applicant", "include", "exclude_reason", "auto_status", "edgar_flags", "auto_mcap_m"]]

## Step 8: Combine and save

In [ ]:
review["final_include"] = np.where(review["include"].isin(["Y", "N"]), review["include"],
    np.where(review["auto_status"] == "include", "Y",
    np.where(review["auto_status"].str.startswith("exclude"), "N", "review")))

review.to_csv(DATA / "processed/research_auto.csv", index=False)
print(review["final_include"].value_counts())

# The remaining manual work:
review[review["final_include"] == "review"][
    ["Proprietary Name", "Applicant", "FDA Approval Date", "auto_ticker", "auto_mcap_m", "auto_status", "edgar_flags"]]